In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

In [2]:
con = duckdb.connect(r'C:\Users\marzieh\Documents\GitHub\Tennis-project\data\tennis.duckdb', read_only=True)
con.execute("SHOW TABLES").df()

,name
0,_build_info
1,_import_audit
2,_schema_audit
3,_source_files
4,game_point_by_point
5,match_away_score
6,match_away_team
7,match_event
8,match_home_score
9,match_home_team


In [57]:
query = """
    SELECT e.match_id,
           h.player_id as home_id,
           h.name as home_name, 
           a.player_id as away_id,
           a.name as away_name, 
           winner_code
    FROM match_home_team as h
    INNER JOIN match_away_team as a
    ON h.match_id = a.match_id
    INNER JOIN match_event as e
    ON e.match_id = h.match_id
    WHERE winner_code IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY e.match_id
        ORDER BY e.match_id
    ) = 1
"""
df = con.execute(query).df()
df

,match_id,home_id,home_name,away_id,away_name,winner_code
0,12019043,161448,Jones B.,197546,Salle F.,2
1,12022248,311586,Agafonov E.,340742,Bertrand R.,1
2,12023362,257637,Pannu K.,53183,Ilkel C.,1
3,11999013,390214,Blockx A.,209103,Molleker R.,2
4,12020510,224033,Naito Y.,134532,You X.,1
...,...,...,...,...,...,...
9603,12210112,52276,Krueger M.,104967,Brouwer G.,2
9604,12210279,92879,Shymanovich I.,67322,Tan H.,1
9605,12211700,135084,Bouquet L.,190877,Guerrieri A.,2
9606,12211784,213081,Delage P.,379585,Safonov A.,2


In [58]:
df = df.drop_duplicates(subset="match_id").reset_index(drop=True)
df

,match_id,home_id,home_name,away_id,away_name,winner_code
0,12019043,161448,Jones B.,197546,Salle F.,2
1,12022248,311586,Agafonov E.,340742,Bertrand R.,1
2,12023362,257637,Pannu K.,53183,Ilkel C.,1
3,11999013,390214,Blockx A.,209103,Molleker R.,2
4,12020510,224033,Naito Y.,134532,You X.,1
...,...,...,...,...,...,...
9603,12210112,52276,Krueger M.,104967,Brouwer G.,2
9604,12210279,92879,Shymanovich I.,67322,Tan H.,1
9605,12211700,135084,Bouquet L.,190877,Guerrieri A.,2
9606,12211784,213081,Delage P.,379585,Safonov A.,2


In [66]:
winner_names = pd.Series(
    df["home_name"].where(df["winner_code"] == 1, df["away_name"])
)
winner_names.value_counts().head(1)

home_name
Popko D.    28
Name: count, dtype: int64